In [2]:
import json
from pathlib import Path
import sys

# Add project root to path to import forecaster
# Handle both cases: running from project root or from research directory
current_dir = Path().resolve()
if current_dir.name == 'research':
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.insert(0, str(project_root))

# Ensure we load the local forecaster package (not a cached/installed one)
import importlib
for mod in ['forecaster.models', 'forecaster']:
    if mod in sys.modules:
        del sys.modules[mod]
forecaster = importlib.import_module('forecaster')
from forecaster.models import MatchData

# Load all JSON files from the male directory
male_dir = project_root / 't20s_json' / 'male'
all_json_files = sorted(male_dir.glob('*.json'))
json_files = all_json_files

print(f"Found {len(all_json_files)} JSON files in male directory; reading {len(json_files)} files")

# Store all match data
all_matches = []

for json_file_path in json_files:
    try:
        with open(json_file_path, 'r') as f:
            json_data = json.load(f)
        
        # Convert to Python objects using Pydantic's model_validate
        match_data = MatchData.model_validate(json_data)
        
        # Append all matches (including 'no result'); filtering happens later
        all_matches.append(match_data)
    except Exception as e:
        print(f"Error processing {json_file_path.name}: {e}")

print(f"\nSuccessfully loaded {len(all_matches)} matches")

# Display info about first match as example
if all_matches:
    match_data = all_matches[0]
    print(f"\n--- Example: First Match ---")
    print(f"Match: {match_data.info.teams[0]} vs {match_data.info.teams[1]}")
    print(f"Date: {match_data.info.dates[0]}")
    print(f"Venue: {match_data.info.venue}")
    # print(f"Winner: {match_data.info.outcome.winner}")
    print(f"Number of innings: {len(match_data.innings)}")
    print(f"Total overs in first innings: {len(match_data.innings[0].overs)}")

Found 3112 JSON files in male directory; reading 3112 files

Successfully loaded 3112 matches

--- Example: First Match ---
Match: Australia vs Sri Lanka
Date: 2017-02-19
Venue: Simonds Stadium, South Geelong
Number of innings: 2
Total overs in first innings: 20


## Data Parse Summery Check

print("=" * 60)
print("SUMMARY")
print("=" * 60)

files_found = len(json_files) if 'json_files' in globals() else 0
matches_loaded = len(all_matches) if 'all_matches' in globals() else 0
errors = files_found - matches_loaded if files_found >= matches_loaded else 0

print(f"Files found: {files_found}")
print(f"Matches loaded: {matches_loaded}")
print(f"Errors: {errors}")

sample_count = min(3, matches_loaded)
if matches_loaded:
    print("\nSample matches:")
    for i in range(sample_count):
        md = all_matches[i]
        print(f"  {i+1}. {md.info.teams[0]} vs {md.info.teams[1]} at {md.info.venue} on {md.info.dates[0]}")

In [3]:
# Filter out matches with outcome "no result"

# Preserve original count before filtering
original_count = len(all_matches) if 'all_matches' in globals() else 0

# Apply filter: keep matches where outcome.result is not "no result"
all_matches_with_result = []
for md in all_matches:
    outcome = getattr(md.info, 'outcome', None)
    result = getattr(outcome, 'result', None)
    if isinstance(result, str) and result.strip().lower() == 'no result':
        continue
    all_matches_with_result.append(md)

excluded_count = original_count - len(all_matches_with_result)

print("=" * 60)
print("FILTER: EXCLUDE 'no result' OUTCOMES")
print("=" * 60)
print(f"Original matches: {original_count}")
print(f"Excluded 'no result': {excluded_count}")
print(f"Filtered matches (all_matches_with_result): {len(all_matches_with_result)}")

# Optional: show a small sample after filtering
sample_count = min(3, len(all_matches_with_result))
if all_matches_with_result:
    print("\nSample filtered matches:")
    for i in range(sample_count):
        md = all_matches_with_result[i]
        print(f"  {i+1}. {md.info.teams[0]} vs {md.info.teams[1]} at {md.info.venue} on {md.info.dates[0]}")

FILTER: EXCLUDE 'no result' OUTCOMES
Original matches: 3112
Excluded 'no result': 74
Filtered matches (all_matches_with_result): 3038

Sample filtered matches:
  1. Australia vs Sri Lanka at Simonds Stadium, South Geelong on 2017-02-19
  2. Australia vs Sri Lanka at Adelaide Oval on 2017-02-22
  3. Ireland vs Hong Kong at Bready Cricket Club, Magheramason on 2016-09-05


In [ ]:
# Check matches where first innings overs < 20 and wickets < 10

# Use filtered list if available; otherwise use all
matches_source = all_matches_with_result if 'all_matches_with_result' in globals() and all_matches_with_result else all_matches

qualified = []  # (md, overs_count, wickets_count)

for md in matches_source:
    if not md.innings:
        continue
    first_innings = md.innings[0]
    overs_count = len(first_innings.overs)
    wickets_count = 0
    for over in first_innings.overs:
        for delivery in over.deliveries:
            if delivery.wickets:
                wickets_count += len(delivery.wickets)
    
    if overs_count < 20 and wickets_count < 10:
        qualified.append((md, overs_count, wickets_count))

print("=" * 60)
print("CHECK: First innings overs < 20 AND wickets < 10")
print("=" * 60)
print(f"Total matches considered: {len(matches_source)}")
print(f"Qualified matches: {len(qualified)}")

# Show a small sample
sample_count = min(10, len(qualified))
if qualified:
    print("\nSample qualified matches:")
    for i in range(sample_count):
        md, oc, wc = qualified[i]
        print(f"  {i+1}. {md.info.teams[0]} vs {md.info.teams[1]} on {md.info.dates[0]} | Overs: {oc}, Wickets: {wc}")

In [4]:
# Exclude matches where first innings overs < 20 and wickets < 10

# Use filtered list if available; otherwise use all
matches_source = all_matches_with_result if 'all_matches_with_result' in globals() and all_matches_with_result else all_matches

excluded = []
kept = []  # will become all_matches_with_result_and_20_overs

for md in matches_source:
    if not md.innings:
        kept.append(md)
        continue
    first_innings = md.innings[0]
    overs_count = len(first_innings.overs)
    wickets_count = 0
    for over in first_innings.overs:
        for delivery in over.deliveries:
            if delivery.wickets:
                wickets_count += len(delivery.wickets)
    
    if overs_count < 20 and wickets_count < 10:
        excluded.append((md, overs_count, wickets_count))
    else:
        kept.append(md)

# Update new working list
all_matches_with_result_and_20_overs = kept

print("=" * 60)
print("FILTER: Keep matches with full/complete first innings")
print("Condition excluded: overs < 20 AND wickets < 10")
print("=" * 60)
print(f"Original matches considered: {len(matches_source)}")
print(f"Excluded (incomplete): {len(excluded)}")
print(f"Kept (all_matches_with_result_and_20_overs): {len(all_matches_with_result_and_20_overs)}")

# Show a small sample of excluded and kept
show_n = 5
if excluded:
    print("\nSample excluded matches:")
    for i in range(min(show_n, len(excluded))):
        md, oc, wc = excluded[i]
        print(f"  {i+1}. {md.info.teams[0]} vs {md.info.teams[1]} on {md.info.dates[0]} | Overs: {oc}, Wickets: {wc}")

if all_matches_with_result_and_20_overs:
    print("\nSample kept matches:")
    for i in range(min(show_n, len(all_matches_with_result_and_20_overs))):
        md = all_matches_with_result_and_20_overs[i]
        print(f"  {i+1}. {md.info.teams[0]} vs {md.info.teams[1]} on {md.info.dates[0]}")

FILTER: Keep matches with full/complete first innings
Condition excluded: overs < 20 AND wickets < 10
Original matches considered: 3038
Excluded (incomplete): 115
Kept (all_matches_with_result_and_20_overs): 2923

Sample excluded matches:
  1. Australia vs India on 2017-10-07 | Overs: 19, Wickets: 8
  2. India vs New Zealand on 2017-11-07 | Overs: 8, Wickets: 5
  3. Sri Lanka vs India on 2018-03-12 | Overs: 19, Wickets: 9
  4. South Africa vs Australia on 2018-11-17 | Overs: 10, Wickets: 6
  5. Australia vs India on 2018-11-21 | Overs: 17, Wickets: 4

Sample kept matches:
  1. Australia vs Sri Lanka on 2017-02-19
  2. Australia vs Sri Lanka on 2017-02-22
  3. Ireland vs Hong Kong on 2016-09-05
  4. Zimbabwe vs India on 2016-06-18
  5. Zimbabwe vs India on 2016-06-20


In [48]:
# Find the latest match in all_matches_with_result_and_20_overs
from datetime import date as _date

# Choose source list
if 'all_matches_with_result_and_20_overs' in globals() and all_matches_with_result_and_20_overs:
    matches_source = all_matches_with_result_and_20_overs
elif 'all_matches_with_result' in globals() and all_matches_with_result:
    matches_source = all_matches_with_result
else:
    matches_source = all_matches

# Helper to parse the latest date from md.info.dates
def _latest_date(md):
    dates = getattr(md.info, 'dates', []) or []
    best = None
    for d in dates:
        try:
            parsed = _date.fromisoformat(d)
            if best is None or parsed > best:
                best = parsed
        except Exception:
            continue
    return best

latest_md = None
latest_dt = None
for md in matches_source:
    d = _latest_date(md)
    if d and (latest_dt is None or d > latest_dt):
        latest_dt = d
        latest_md = md

print("=" * 60)
print("LATEST MATCH (from filtered list)")
print("=" * 60)
if latest_md:
    print(f"Date: {latest_dt}")
    print(f"Match: {latest_md.info.teams[0]} vs {latest_md.info.teams[1]}")
    print(f"Venue: {latest_md.info.venue}")
else:
    print("No matches available to determine latest.")

LATEST MATCH (from filtered list)
Date: 2026-01-07
Match: Sri Lanka vs Pakistan
Venue: Rangiri Dambulla International Stadium


In [38]:
# Break match_data into two innings and calculate scores

# Extract the two innings
first_innings = match_data.innings[0]
second_innings = match_data.innings[1]



In [39]:
# Enhanced view: Show runs scored in each section (not just cumulative)

def get_innings_by_sections_with_runs_per_section(innings, total_overs, balls_per_over, section_size=2):
    """
    Break innings into sections and show both cumulative and runs per section.
    Adds balls remaining and boundary counts at the end of each section.
    """
    sections = []
    cumulative_runs = 0
    cumulative_wickets = 0
    cumulative_overs = 0
    current_section_overs = []
    section_runs = 0
    section_wickets = 0
    section_overs = 0
    cumulative_fours = 0
    cumulative_sixes = 0
    cumulative_dot_balls = 0
    section_fours = 0
    section_sixes = 0
    section_dot_balls = 0
    
    for over in innings.overs:
        # Process all deliveries in this over
        for delivery in over.deliveries:
            runs_in_delivery = delivery.runs.total
            cumulative_runs += runs_in_delivery
            section_runs += runs_in_delivery
            
            if runs_in_delivery == 0:
                cumulative_dot_balls += 1
                section_dot_balls += 1
            
            if delivery.runs.batter == 4:
                cumulative_fours += 1
                section_fours += 1
            elif delivery.runs.batter == 6:
                cumulative_sixes += 1
                section_sixes += 1
            
            if delivery.wickets:
                wickets_in_delivery = len(delivery.wickets)
                cumulative_wickets += wickets_in_delivery
                section_wickets += wickets_in_delivery
        
        current_section_overs.append(over.over)
        cumulative_overs += 1
        section_overs += 1
        
        # Check if we've completed a section (2 overs)
        if len(current_section_overs) == section_size:
            # Calculate run rates: runs / overs
            cumulative_run_rate = (cumulative_runs / cumulative_overs) if cumulative_overs > 0 else 0
            section_run_rate = (section_runs / section_overs) if section_overs > 0 else 0
            
            # Powerplay is first 6 overs (0-5), so powerplay_completed is True if overs_completed > 6
            powerplay_completed = (current_section_overs[-1] + 1) > 6
            
            overs_completed = current_section_overs[-1] + 1
            balls_remaining = max(total_overs - overs_completed, 0) * balls_per_over
            boundaries_last_2_overs = section_fours + section_sixes
            
            section_num = len(sections) + 1
            sections.append({
                'section': section_num,
                'overs': f"{current_section_overs[0]}-{current_section_overs[-1]}",
                'overs_completed': overs_completed,
                'runs_in_section': section_runs,
                'wickets_in_section': section_wickets,
                'section_run_rate': section_run_rate,
                'cumulative_runs': cumulative_runs,
                'cumulative_wickets': cumulative_wickets,
                'cumulative_run_rate': cumulative_run_rate,
                'powerplay_completed': powerplay_completed,
                'balls_remaining': balls_remaining,
                'fours_hit': cumulative_fours,
                'sixes_hit': cumulative_sixes,
                'boundaries_last_2_overs': boundaries_last_2_overs,
                'dot_balls_so_far': cumulative_dot_balls,
                'dot_balls_last_2_overs': section_dot_balls
            })
            current_section_overs = []
            section_runs = 0
            section_wickets = 0
            section_overs = 0
            section_fours = 0
            section_sixes = 0
            section_dot_balls = 0
    
    # Handle remaining overs
    if current_section_overs:
        cumulative_run_rate = (cumulative_runs / cumulative_overs) if cumulative_overs > 0 else 0
        section_run_rate = (section_runs / section_overs) if section_overs > 0 else 0
        powerplay_completed = (current_section_overs[-1] + 1) > 6
        overs_completed = current_section_overs[-1] + 1
        balls_remaining = max(total_overs - overs_completed, 0) * balls_per_over
        boundaries_last_2_overs = section_fours + section_sixes
        section_num = len(sections) + 1
        sections.append({
            'section': section_num,
            'overs': f"{current_section_overs[0]}-{current_section_overs[-1]}",
            'overs_completed': overs_completed,
            'runs_in_section': section_runs,
            'wickets_in_section': section_wickets,
            'section_run_rate': section_run_rate,
            'cumulative_runs': cumulative_runs,
            'cumulative_wickets': cumulative_wickets,
            'cumulative_run_rate': cumulative_run_rate,
            'powerplay_completed': powerplay_completed,
            'balls_remaining': balls_remaining,
            'fours_hit': cumulative_fours,
            'sixes_hit': cumulative_sixes,
            'boundaries_last_2_overs': boundaries_last_2_overs,
            'dot_balls_so_far': cumulative_dot_balls,
            'dot_balls_last_2_overs': section_dot_balls
        })
    
    return sections

# Get enhanced sections for both innings
first_innings_detailed = get_innings_by_sections_with_runs_per_section(
    first_innings, match_data.info.overs, match_data.info.balls_per_over
)
second_innings_detailed = get_innings_by_sections_with_runs_per_section(
    second_innings, match_data.info.overs, match_data.info.balls_per_over
)

# Display enhanced view
print("\n" + "=" * 130)
print("DETAILED SECTION BREAKDOWN (Runs per section + Cumulative + Run Rates)")
print("=" * 130)

for team_name, sections, innings_num in [
    (first_innings.team, first_innings_detailed, "1st"),
    (second_innings.team, second_innings_detailed, "2nd")
]:
    print(f"\n{team_name} ({innings_num} Innings):")
    print(f"{'Section':<10} {'Overs':<15} {'Runs':<8} {'Wkts':<6} {'RR':<8} {'Cum Runs':<12} {'Cum Wkts':<12} {'Cum RR':<10} {'PP Done':<10} {'Balls Rem':<10} {'Fours':<7} {'Sixes':<7} {'Bnds L2':<8} {'Dot so far':<12} {'Dot L2':<8}")
    print("-" * 170)
    for section in sections:
        print(f"{section['section']:<10} {section['overs']:<15} "
              f"{section['runs_in_section']:<8} {section['wickets_in_section']:<6} "
              f"{section['section_run_rate']:<8.2f} "
              f"{section['cumulative_runs']:<12} {section['cumulative_wickets']:<12} "
              f"{section['cumulative_run_rate']:<10.2f} {str(section['powerplay_completed']):<10} {section['balls_remaining']:<10} "
              f"{section['fours_hit']:<7} {section['sixes_hit']:<7} {section['boundaries_last_2_overs']:<8} "
              f"{section['dot_balls_so_far']:<12} {section['dot_balls_last_2_overs']:<8}")

print("\n" + "=" * 130)



DETAILED SECTION BREAKDOWN (Runs per section + Cumulative + Run Rates)

Australia (1st Innings):
Section    Overs           Runs     Wkts   RR       Cum Runs     Cum Wkts     Cum RR     PP Done    Balls Rem  Fours   Sixes   Bnds L2  Dot so far   Dot L2  
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
1          0-1             17       0      8.50     17           0            8.50       False      108        2       0       2        4            4       
2          2-3             13       1      6.50     30           1            7.50       False      96         4       0       2        11           7       
3          4-5             30       0      15.00    60           1            10.00      False      84         6       3       5        15           4       
4          6-7             7        1      3.50     67           2            8.38       True      

In [49]:
# Average first-innings final score per venue (filtered list)

# Choose source list
if 'all_matches_with_result_and_20_overs' in globals() and all_matches_with_result_and_20_overs:
    matches_source = all_matches_with_result_and_20_overs
elif 'all_matches_with_result' in globals() and all_matches_with_result:
    matches_source = all_matches_with_result
else:
    matches_source = all_matches

from collections import defaultdict

def first_innings_total(md):
    if not md.innings:
        return None
    first_innings = md.innings[0]
    total = 0
    for over in first_innings.overs:
        for delivery in over.deliveries:
            total += delivery.runs.total
    return total

venue_stats = defaultdict(lambda: {"sum": 0, "count": 0})

for md in matches_source:
    venue = getattr(md.info, 'venue', None) or 'Unknown Venue'
    total = first_innings_total(md)
    if total is None:
        continue
    venue_stats[venue]["sum"] += total
    venue_stats[venue]["count"] += 1

results = []  # (venue, matches, average)
for venue, sc in venue_stats.items():
    if sc["count"] > 0:
        avg = sc["sum"] / sc["count"]
        results.append((venue, sc["count"], avg))

# Sort by average descending
results.sort(key=lambda x: x[2], reverse=True)

print("=" * 60)
print("AVERAGE FIRST-INNINGS TOTAL PER VENUE")
print("=" * 60)
print(f"Total venues: {len(results)} | Matches considered: {len(matches_source)}")

# Show top 20 venues by average
top_n = 20
print(f"\nTop {top_n} venues by average first-innings total:")
for i, (venue, cnt, avg) in enumerate(results[:top_n], start=1):
    print(f"  {i}. {venue}: avg={avg:.2f} over {cnt} matches")

# Optionally, show a few lowest averages
print("\nBottom 10 venues by average first-innings total:")
for i, (venue, cnt, avg) in enumerate(results[-10:], start=max(len(results)-9,1)):
    print(f"  {i}. {venue}: avg={avg:.2f} over {cnt} matches")

AVERAGE FIRST-INNINGS TOTAL PER VENUE
Total venues: 322 | Matches considered: 2923

Top 20 venues by average first-innings total:
  1. Rajiv Gandhi International Stadium, Uppal, Hyderabad: avg=241.50 over 2 matches
  2. Narendra Modi Stadium, Ahmedabad: avg=232.50 over 2 matches
  3. Barsapara Cricket Stadium, Guwahati: avg=229.50 over 2 matches
  4. Holkar Cricket Stadium, Indore: avg=227.00 over 1 matches
  5. University Oval: avg=219.00 over 1 matches
  6. Old Trafford, Manchester: avg=218.67 over 3 matches
  7. Arun Jaitley Stadium, Delhi: avg=216.00 over 2 matches
  8. Maharaja Yadavindra Singh International Cricket Stadium, New Chandigarh: avg=213.00 over 1 matches
  9. National Cricket Stadium, Grenada: avg=208.00 over 1 matches
  10. Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh: avg=208.00 over 1 matches
  11. Trent Bridge, Nottingham: avg=207.33 over 3 matches
  12. Rajiv Gandhi International Stadium, Uppal: avg=207.00 over 1 matches
  13. The Wanderers Sta

In [6]:
# Prepare per-venue average first-innings totals (no file write here)
import json
from collections import defaultdict
from pathlib import Path

# Choose source list
if 'all_matches_with_result_and_20_overs' in globals() and all_matches_with_result_and_20_overs:
    matches_source = all_matches_with_result_and_20_overs
elif 'all_matches_with_result' in globals() and all_matches_with_result:
    matches_source = all_matches_with_result
else:
    matches_source = all_matches

# Compute first-innings total per match
def first_innings_total(md):
    if not md.innings:
        return None
    first_innings = md.innings[0]
    total = 0
    for over in first_innings.overs:
        for delivery in over.deliveries:
            total += delivery.runs.total
    return total

venue_stats = defaultdict(lambda: {"sum": 0, "count": 0})

for md in matches_source:
    venue = getattr(md.info, 'venue', None) or 'Unknown Venue'
    total = first_innings_total(md)
    if total is None:
        continue
    venue_stats[venue]["sum"] += total
    venue_stats[venue]["count"] += 1

# Build average dataset: list of {venue, matches, average_first_innings_total}
venue_avg_data = []
for venue, sc in venue_stats.items():
    if sc["count"] > 0:
        avg = sc["sum"] / sc["count"]
        venue_avg_data.append({
            "venue": venue,
            "matches": int(sc["count"]),
            "average_first_innings_total": round(avg, 2)
        })

# Sort by average descending for convenience
venue_avg_data.sort(key=lambda x: x["average_first_innings_total"], reverse=True)

print("Prepared average first-innings totals for:", len(venue_avg_data), "venues")
print("Matches considered:", len(matches_source))

Prepared average first-innings totals for: 322 venues
Matches considered: 2923


In [7]:
# Compute par scores, merge with averages, and save single combined JSON
import pandas as pd
import numpy as np
from pathlib import Path
import json

# Ensure matches_source is available
if 'matches_source' not in globals() or matches_source is None:
    if 'all_matches_with_result_and_20_overs' in globals() and all_matches_with_result_and_20_overs:
        matches_source = all_matches_with_result_and_20_overs
    elif 'all_matches_with_result' in globals() and all_matches_with_result:
        matches_source = all_matches_with_result
    else:
        matches_source = all_matches

# Helper: first-innings total (redefine for safety)
def first_innings_total(md):
    if not md.innings:
        return None
    first_innings = md.innings[0]
    total = 0
    for over in first_innings.overs:
        for delivery in over.deliveries:
            total += delivery.runs.total
    return total

# Par score calculation (windowed win% approach)
def calculate_par_score(venue_df: pd.DataFrame, target_win_pct: float = 0.52):
    # Expect columns: ['1st_innings_score', 'batting_first_won']
    if venue_df.empty:
        return None
    sorted_data = venue_df.sort_values('1st_innings_score')
    scores = []
    win_percentages = []
    for score in range(120, 221, 5):
        window = sorted_data[(sorted_data['1st_innings_score'] >= score - 5) &
                             (sorted_data['1st_innings_score'] <= score + 5)]
        if len(window) >= 3:
            win_pct = window['batting_first_won'].mean()
            scores.append(score)
            win_percentages.append(win_pct)
    if not win_percentages:
        return None
    win_pct_array = np.array(win_percentages)
    closest_idx = int(np.argmin(np.abs(win_pct_array - target_win_pct)))
    return int(scores[closest_idx])

# Build per-venue rows for par calculation
venue_rows = {}  # venue -> list of rows {'1st_innings_score', 'batting_first_won'}
for md in matches_source:
    venue = getattr(md.info, 'venue', None) or 'Unknown Venue'
    total = first_innings_total(md)
    if total is None:
        continue
    first_team = md.innings[0].team if md.innings else None
    outcome = getattr(md.info, 'outcome', None)
    winner = getattr(outcome, 'winner', None) if outcome is not None else None
    if winner is None or first_team is None:
        # Skip if we can't determine win/loss
        continue
    batting_first_won = 1 if winner == first_team else 0
    venue_rows.setdefault(venue, []).append({'1st_innings_score': total, 'batting_first_won': batting_first_won})

# Compute par scores per venue and map
target_win_pct = 0.52
par_map = {}
for venue, rows in venue_rows.items():
    df = pd.DataFrame(rows)
    par = calculate_par_score(df, target_win_pct=target_win_pct)
    par_map[venue] = {
        'par_score': par,
        'matches_for_par': int(len(df))
    }

# Merge averages with par scores
if 'venue_avg_data' not in globals():
    # Fallback: recompute minimal averages if Cell 10 wasn't run
    from collections import defaultdict
    venue_stats = defaultdict(lambda: {"sum": 0, "count": 0})
    for md in matches_source:
        venue = getattr(md.info, 'venue', None) or 'Unknown Venue'
        total = first_innings_total(md)
        if total is None:
            continue
        venue_stats[venue]["sum"] += total
        venue_stats[venue]["count"] += 1
    venue_avg_data = []
    for venue, sc in venue_stats.items():
        if sc["count"] > 0:
            avg = sc["sum"] / sc["count"]
            venue_avg_data.append({
                "venue": venue,
                "matches": int(sc["count"]),
                "average_first_innings_total": round(avg, 2)
            })

# Combine into single structure
combined_venues = []
for item in venue_avg_data:
    venue = item['venue']
    par_entry = par_map.get(venue, {})
    combined_venues.append({
        'venue': venue,
        'matches': item['matches'],
        'average_first_innings_total': item['average_first_innings_total'],
        'par_score': par_entry.get('par_score')
    })

# Sort by average or par as desired (keep average desc)
combined_venues.sort(key=lambda x: x['average_first_innings_total'], reverse=True)

# Save to single JSON in t20s_json
out_path = project_root / 't20s_json' / 'venue_metrics.json'
with open(out_path, 'w') as f:
    json.dump({
        'source_matches': len(matches_source),
        'target_win_pct': target_win_pct,
        'venues': combined_venues
    }, f, indent=2)

print('Saved combined venue metrics to:', out_path)
print('Venues with averages:', len(venue_avg_data))
print('Venues with par scores:', sum(1 for v in combined_venues if v['par_score'] is not None))

Saved combined venue metrics to: /Users/damith/projects/t20_score_prediction/t20s_json/venue_metrics.json
Venues with averages: 322
Venues with par scores: 104
